## What Is an LLM and How It Generates Text

A Large Language Model is a neural network trained to predict the next token in a sequence of text, given everything that came before it.
Every single thing an LLM produces — an answer, a story, code, a translation — comes from repeating this one operation: predict the next token, add it, repeat.

## How it actually works

**The core mechanism — autoregressive generation:**
An LLM does not "know" the answer to your question and write it out directly.
It generates text one token at a time, where each new token is chosen based on
everything generated so far — including the tokens it just generated itself.

Step by step:
1. You provide a prompt (your input text)
2. The model looks at the entire sequence so far and calculates a probability
   distribution over every possible next token — essentially "how likely is each
   possible next word/piece, given everything before it"
3. A token is selected (the exact selection process involves sampling — covered
   in the next notebook on temperature)
4. That token is appended to the sequence
5. The model repeats the whole process again, now including the token it just added
6. This continues until the model produces a special "stop" token, or a length limit is hit

**Why this means LLMs are fundamentally prediction machines, not lookup machines:**
There is no database of pre-written answers being searched. Every response is
generated fresh, token by token, based on patterns learned during training.
This is also why LLMs can produce genuinely novel text that has never existed before —
they are not retrieving, they are predicting and constructing.

**What "training" actually optimised for:**
During training, the model was shown enormous amounts of text and adjusted its
internal parameters (covered in ml-fundamentals) to get better and better at
predicting the next token in real text — like predicting the next word in a sentence
from a book, an article, code, or a conversation. Nothing more exotic than that
single objective produces everything an LLM can do.

**Why a model trained on "just predict the next token" can do things like reasoning:**
This is genuinely one of the more remarkable findings in the field. Predicting the
next token well, across an enormous and diverse range of text, apparently requires
the model to develop internal representations of grammar, facts, relationships,
and even something resembling step-by-step reasoning — because text that reasons
correctly is, statistically, more predictable than text that reasons incorrectly,
once the model has seen enough examples of both.

## Where it breaks or gets dangerous

**The model has no concept of truth, only likelihood:**
Because the entire mechanism is "predict the most statistically likely next token,"
the model has no internal "is this actually true" check built into the core process.
If a false statement is more likely-sounding given the patterns it learned than a
true one in a specific context, it can produce the false one with identical confidence
to a true one. This is the root cause of hallucination (covered fully in ml-fundamentals,
notebook 12) — it traces directly back to this core mechanism.

**Errors can compound across the generation process:**
Because each new token depends on everything generated so far — including the model's
OWN previous tokens — an early mistake can shape everything that follows. If the model
generates a slightly wrong premise early in a response, subsequent tokens are predicted
based on a sequence that now CONTAINS that wrong premise, making the model more likely
to continue building on the error rather than catching it.

**The "stochastic" part matters — this is not the same model output every time:**
Because the next-token selection involves sampling rather than always picking the single
most likely token, the exact same prompt can produce different exact wording across runs
(this connects directly to "deterministic vs non-deterministic" — covered later in this folder).
This is not a flaw being patched — it is inherent to how generation works, and it is why
testing LLM-based systems requires different approaches than testing traditional deterministic code.

**Common misconception worth correcting directly:**
People sometimes describe LLMs as "just autocomplete" dismissively, as if this makes
the capability trivial. The mechanism IS fundamentally next-token prediction — that part
is accurate — but treating this as proof of limited capability misses that sufficiently
good next-token prediction, at sufficient scale, produces emergent abilities (covered in
ml-fundamentals notebook 25) that go well beyond what "autocomplete" implies casually.
The mechanism is simple to describe. The resulting behaviour at scale is not simple at all.

In [ ]:
# What Is an LLM — illustrating the autoregressive generation loop conceptually

import anthropic
import os
from dotenv import load_dotenv

load_dotenv()
client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

print("=== Standard generation: one request, full response returned at once ===")
response = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=50,
    messages=[{"role": "user", "content": "Complete this sentence: The three-way match in D365 compares"}]
)
print(response.content[0].text)
print()

print("=== Streaming generation: watching the token-by-token process directly ===")
print("(This makes the autoregressive nature visible — each piece appears as it is generated)")
print()

with client.messages.stream(
    model="claude-haiku-4-5",
    max_tokens=50,
    messages=[{"role": "user", "content": "Complete this sentence: The three-way match in D365 compares"}]
) as stream:
    for text in stream.text_stream:
        print(text, end="", flush=True)

print()
print()
print("Each piece you saw appear was generated based on everything before it —")
print("including the pieces the model itself had already produced in this same response.")
print("This is the autoregressive loop happening in real time, not a typing animation.")

## My D365 analogy

An LLM generating text is similar to a consultant filling in a complex D365 workflow
approval comment field, one word at a time, where each word choice is influenced by
every word written so far in that same comment.

The consultant is not retrieving a pre-written comment from a database that exactly
matches this exact invoice situation. They are constructing something fresh, word by word,
based on patterns from thousands of similar comments they have written or read before —
"if the invoice amount exceeds tolerance, the next word is usually 'escalating' or
'routing'," shaped by everything written in the sentence so far.

If the consultant starts the comment with a slightly wrong assumption — say, misreading
which legal entity is involved — everything written afterward tends to build on that
wrong assumption, because each subsequent word is chosen based on a sentence that now
already contains the error. Catching it requires either noticing mid-sentence and
correcting, or reviewing the whole thing afterward — not something that happens
automatically just because more words get added.

The pattern-completion instinct itself is not wrong or unusual — it is how fluent,
context-aware writing normally works, for humans and for these models. The risk
is treating fluent, confident-sounding pattern completion as automatically meaning
factually verified.

## LinkedIn post idea

Every single thing a language model produces — an essay, a line of code, an answer
to your question — comes from doing the exact same operation, over and over:
predict the single most likely next piece of text, add it, then do it again.

That is genuinely it. There is no separate "thinking module" bolted onto a "writing module."
The entire system is one operation, repeated.

What surprised me learning this properly: that simple, repeated operation — done at a
large enough scale, on a large enough amount of text — appears to require the model to
build something that looks a lot like genuine understanding of grammar, facts, and
even multi-step reasoning, just to get good at predicting what word comes next.

It also explains exactly why these models can sound completely confident while being
completely wrong. The mechanism has no separate "is this true" checkpoint built in.
It only has "is this the statistically likely next piece of text," and those two
things are usually — but not always — the same thing.

Fourteen years of writing approval comments taught me that fluent and correct are not
the same property. Apparently that lesson applies to language models too.